In [ ]:
# Thiết lập cho Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 4 - Các Kỹ thuật Phân loại (Classification Techniques)

Yêu cầu:
1. **K-Nearest Neighbors, Naive Bayes:** Triển khai KNN. Thử nghiệm thay đổi K và quan sát Decision Boundary. Xây dựng bộ lọc Spam bằng Naive Bayes.
2. **Decision Tree:** Phân loại bệnh nhân (Sử dụng tập dữ liệu Ung thư vú - Breast Cancer). Trực quan hóa cây quyết định.
3. **Support Vector Machines (SVM):** Phân loại dữ liệu phi tuyến (hình mặt trăng - `make_moons`) dùng SVM với Kernel RBF.

## 1. K-Nearest Neighbors (KNN) & Naive Bayes

### 1.1 Triển khai KNN và Đường biên quyết định (Decision Boundary)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier

# Tạo dữ liệu
X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

def plot_decision_boundary(clf, X, y, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_new = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_new).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
    plt.axis(axes)
    plt.xlabel(r"$x_1$", fontsize=14)
    plt.ylabel(r"$x_2$", fontsize=14, rotation=0)

plt.figure(figsize=(11, 4))

for i, k in enumerate([1, 5, 30]):
    plt.subplot(131 + i)
    knn_clf = KNeighborsClassifier(n_neighbors=k)
    knn_clf.fit(X, y)
    plot_decision_boundary(knn_clf, X, y, [-1.5, 2.5, -1, 1.5])
    plt.title("k = {}".format(k), fontsize=16)

plt.tight_layout()
plt.show()

### 1.2 Xây dựng bộ lọc Thư rác (Spam Filter) dùng Naive Bayes

In [ ]:
import os
import tarfile
import urllib.request

DOWNLOAD_ROOT = "http://spamassassin.apache.org/old/publiccorpus/"
HAM_URL = DOWNLOAD_ROOT + "20030228_easy_ham.tar.bz2"
SPAM_URL = DOWNLOAD_ROOT + "20030228_spam.tar.bz2"
SPAM_PATH = os.path.join("datasets", "spam")

def fetch_spam_data(ham_url=HAM_URL, spam_url=SPAM_URL, spam_path=SPAM_PATH):
    if not os.path.isdir(spam_path):
        os.makedirs(spam_path)
    for filename, url in (("ham.tar.bz2", ham_url), ("spam.tar.bz2", spam_url)):
        path = os.path.join(spam_path, filename)
        if not os.path.isfile(path):
            urllib.request.urlretrieve(url, path)
        tar_bz2_file = tarfile.open(path)
        tar_bz2_file.extractall(path=spam_path)
        tar_bz2_file.close()

# Tải dữ liệu thư rác
fetch_spam_data()

HAM_DIR = os.path.join(SPAM_PATH, "easy_ham")
SPAM_DIR = os.path.join(SPAM_PATH, "spam")
ham_filenames = [name for name in sorted(os.listdir(HAM_DIR)) if len(name) > 20]
spam_filenames = [name for name in sorted(os.listdir(SPAM_DIR)) if len(name) > 20]

import email
import email.policy

def load_email(is_spam, filename, spam_path=SPAM_PATH):
    directory = "spam" if is_spam else "easy_ham"
    with open(os.path.join(spam_path, directory, filename), "rb") as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)

ham_emails = [load_email(is_spam=False, filename=name) for name in ham_filenames]
spam_emails = [load_email(is_spam=True, filename=name) for name in spam_filenames]

In [ ]:
import re
from html import unescape

# Trích xuất văn bản từ HTML
def html_to_plain_text(html):
    text = re.sub('<head.*?>.*?</head>', '', html, flags=re.M | re.S | re.I)
    text = re.sub('<a\\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)
    text = re.sub('<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\\s*\\n)+', '\\n', text, flags=re.M | re.S)
    return unescape(text)

def email_to_text(email):
    html = None
    for part in email.walk():
        ctype = part.get_content_type()
        if not ctype in ("text/plain", "text/html"):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if ctype == "text/plain":
            return content
        else:
            html = content
    if html:
        return html_to_plain_text(html)

In [ ]:
try:
    import nltk
    stemmer = nltk.PorterStemmer()
except ImportError:
    stemmer = None

from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# Pipeline tiền xử lý văn bản (Đếm tần suất từ)
class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=True, lower_case=True, remove_punctuation=True,
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        X_transformed = []
        for email in X:
            text = email_to_text(email) or ""
            if self.lower_case:
                text = text.lower()
            if self.replace_numbers:
                text = re.sub(r'\\d+(?:\\.\\d*)?(?:[eE][+-]?\\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count
                word_counts = stemmed_word_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed)

from scipy.sparse import csr_matrix

# Chuyển đổi bộ đếm từ thành Vector đặc trưng
class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size
    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common()[:self.vocabulary_size]
        self.vocabulary_ = {word: index + 1 for index, (word, count) in enumerate(most_common)}
        return self
    def transform(self, X, y=None):
        rows = []
        cols = []
        data = []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)), shape=(len(X), self.vocabulary_size + 1))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score

X = np.array(ham_emails + spam_emails, dtype=object)
y = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocess_pipeline = Pipeline([
    ("email_to_wordcount", EmailToWordCounterTransformer()),
    ("wordcount_to_vector", WordCounterToVectorTransformer()),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

# Khởi tạo và đánh giá Naive Bayes
nb_clf = MultinomialNB()
score = cross_val_score(nb_clf, X_train_transformed, y_train, cv=3)
print("Điểm số trung bình (Cross-Validation) của Naive Bayes: {:.2f}%".format(score.mean() * 100))

## 2. Cây quyết định (Decision Tree)

### 2.1 Phân loại bệnh nhân (Tập dữ liệu Ung thư vú - Breast Cancer) và Trực quan hóa cây

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Tải tập dữ liệu ung thư vú để phân loại bệnh nhân
cancer = load_breast_cancer()
# Lấy 2 đặc trưng đầu tiên để dễ trực quan hóa (tùy chọn, ở đây lấy hết hoặc 2 cái để đơn giản)
# Chúng ta sẽ lấy 'mean radius' và 'mean texture'
X_cancer = cancer.data[:, :2]
y_cancer = cancer.target

tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_cancer, y_cancer)

plt.figure(figsize=(14, 10))
plot_tree(tree_clf, 
          feature_names=cancer.feature_names[:2],  
          class_names=cancer.target_names,
          filled=True,
          rounded=True)
plt.title("Trực quan hóa Cây Quyết Định (Tập dữ liệu Phân loại Bệnh nhân)")
plt.show()

## 3. Support Vector Machines (SVM)

### 3.1 Phân loại phi tuyến trên tập `make_moons` sử dụng Kernel RBF

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_m, y_m = make_moons(n_samples=100, noise=0.15, random_state=42)

rbf_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001))
    ])
rbf_kernel_svm_clf.fit(X_m, y_m)

def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_new = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_new).reshape(x0.shape)
    y_decision = clf.decision_function(X_new).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.contourf(x0, x1, y_decision, cmap=plt.cm.brg, alpha=0.1)

plt.figure(figsize=(8, 5))
plot_predictions(rbf_kernel_svm_clf, [-1.5, 2.5, -1, 1.5])
plt.plot(X_m[:, 0][y_m==0], X_m[:, 1][y_m==0], "bs")
plt.plot(X_m[:, 0][y_m==1], X_m[:, 1][y_m==1], "g^")
plt.title(r"$\gamma=5, C=0.001$", fontsize=16)
plt.axis([-1.5, 2.5, -1, 1.5])
plt.show()

In [ ]:
# Thử nghiệm với các giá trị gamma và C khác nhau
gamma1, gamma2 = 0.1, 5
C1, C2 = 0.001, 1000
hyperparams = (gamma1, C1), (gamma1, C2), (gamma2, C1), (gamma2, C2)

svm_clfs = []
for gamma, C in hyperparams:
    svm_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=gamma, C=C))
        ])
    svm_pipeline.fit(X_m, y_m)
    svm_clfs.append(svm_pipeline)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 7), sharex=True, sharey=True)

for i, svm_clf in enumerate(svm_clfs):
    plt.sca(axes[i // 2, i % 2])
    plot_predictions(svm_clf, [-1.5, 2.5, -1, 1.5])
    plt.plot(X_m[:, 0][y_m==0], X_m[:, 1][y_m==0], "bs")
    plt.plot(X_m[:, 0][y_m==1], X_m[:, 1][y_m==1], "g^")
    gamma, C = hyperparams[i]
    plt.title(r"$\gamma={}, C={}$".format(gamma, C), fontsize=16)
    if i in (0, 1):
        plt.xlabel("")
    if i in (1, 3):
        plt.ylabel("")

plt.tight_layout()
plt.show()